# FAOSTAT bilateral trade reporting quality

Technical companion to the bespoke food-trade Sankey visualization.
Documents the data-quality characteristics of FAOSTAT's *Detailed Trade
Matrix* (TM): how complete the bilateral coverage is, and how closely the
two reports (from the exporting and importing country) agree when both
sides report the same flow.

🚧 **Placeholder — analysis to be filled in.** The plotting helpers are
defined inline below; each section's prose will be fleshed out once the
export step has been validated end-to-end.

## 1. Setup

In [7]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from owid.catalog import Dataset

from etl.paths import DATA_DIR

ds = Dataset(DATA_DIR / "garden" / "faostat" / "2026-05-07" / "faostat_tm")
tb = ds.read("faostat_tm", safe_types=False)
print(f"{len(tb):,} rows; years {int(tb['year'].min())}–{int(tb['year'].max())}")

52,410,630 rows; years 1986–2024


## 2. Year coverage

Rows per year and the number of distinct reporting countries per year.
When both bars drop together, that year is partially reported (typical of
the most recent year, as FAOSTAT publishes with a lag). When only the
rows bar drops, it indicates an actual trade contraction.

In [8]:
def plot_coverage(tb):
    """Show two side-by-side bars per year: number of rows and number of
    distinct reporting countries. Together they distinguish 'less data
    actually reported' (both drop) from 'less trade activity' (rows drop
    but reporter count stays roughly flat) — useful when picking the
    latest well-covered year."""
    grouped = tb.groupby("year", observed=True)
    rows = grouped.size().sort_index()
    reporters = grouped["reporter_country"].nunique().sort_index()
    years = rows.index.astype(int).tolist()

    fig = go.Figure()
    fig.add_bar(x=years, y=rows.values, name="Rows", yaxis="y1", opacity=0.8)
    fig.add_bar(x=years, y=reporters.values, name="Distinct reporters", yaxis="y2", opacity=0.8)
    fig.update_layout(
        title="Detailed trade matrix dataset - Coverage",
        xaxis=dict(title="Year"),
        yaxis=dict(title="Rows", side="left"),
        yaxis2=dict(title="Distinct reporters", side="right", overlaying="y", showgrid=False),
        barmode="group",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()


plot_coverage(tb)

## 3. Reporting coverage by year

For every `(reporter, partner, item, year)` flow we check who reported it:
both sides ("matched"), only the exporter ("exporter-only"), or only the
importer ("importer-only"). The chart shows the share of each bucket per
year — a coverage signal, distinct from quantity agreement.

In [9]:
def plot_reporting_coverage_by_year(tb):
    """Stacked bar chart per year showing what fraction of bilateral flows
    are reported by both sides ("matched"), only by the exporter, or only by
    the importer."""
    qty = tb[tb["element"].isin(["Export quantity", "Import quantity"])][
        ["reporter_country", "partner_country", "item_code", "year", "element"]
    ].copy()
    for col in ("reporter_country", "partner_country"):
        qty[col] = qty[col].astype(str)

    exp_keys = qty.loc[
        qty["element"] == "Export quantity",
        ["reporter_country", "partner_country", "item_code", "year"],
    ].drop_duplicates()
    imp_keys = (
        qty.loc[
            qty["element"] == "Import quantity",
            ["reporter_country", "partner_country", "item_code", "year"],
        ]
        .rename(columns={"reporter_country": "partner_country", "partner_country": "reporter_country"})
        .drop_duplicates()
    )

    merged = exp_keys.merge(
        imp_keys,
        how="outer",
        indicator=True,
        on=["reporter_country", "partner_country", "item_code", "year"],
    )
    by_year = (
        merged.groupby("year", observed=True)["_merge"]
        .value_counts(normalize=True)
        .unstack(fill_value=0.0)
        .rename(columns={"both": "matched", "left_only": "exporter-only", "right_only": "importer-only"})
    )
    by_year = by_year[["matched", "exporter-only", "importer-only"]].reset_index()
    long = by_year.melt(id_vars="year", var_name="status", value_name="share")

    fig = px.bar(
        long,
        x="year",
        y="share",
        color="status",
        title="Reporting coverage",
        labels={"year": "Year", "share": "Share of (reporter, partner, item) tuples"},
        category_orders={"status": ["matched", "exporter-only", "importer-only"]},
    )
    fig.update_layout(barmode="stack", yaxis_tickformat=".0%")
    fig.show()


plot_reporting_coverage_by_year(tb)

## 4. Quantity agreement among matched flows

For matched flows, how closely do the two reported quantities agree? The
agreement ratio is `min(exp_qty, imp_qty) / max(exp_qty, imp_qty)` —
100% means a perfect match, 50% means one side reports twice the other.
The chart bins each reporter's matched flows into five bands and shows
the worst reporters at the top.

Pass `include_unmatched=True` to additionally tag flows reported by only
one side as "Unmatched" (a sixth band, darker red) — captures both
coverage and quantity-agreement problems in one view.

In [10]:
def plot_quantity_mismatch_by_reporter(tb, year=None, top_n=20, include_unmatched=False):
    """Stacked horizontal bar showing how bilateral flows distribute across
    quantity-agreement bands, for the top-N reporting countries."""
    if year is None:
        rows_per_year = tb.groupby("year", observed=True).size()
        year = int(rows_per_year[rows_per_year >= 0.9 * rows_per_year.max()].index.max())

    qty = tb[(tb["year"] == year) & tb["element"].isin(["Export quantity", "Import quantity"])][
        ["reporter_country", "partner_country", "item_code", "element", "value"]
    ].copy()
    for col in ("reporter_country", "partner_country"):
        qty[col] = qty[col].astype(str)

    exp = qty.loc[
        qty["element"] == "Export quantity",
        ["reporter_country", "partner_country", "item_code", "value"],
    ].rename(columns={"value": "exp_qty"})
    imp = qty.loc[
        qty["element"] == "Import quantity",
        ["reporter_country", "partner_country", "item_code", "value"],
    ].rename(
        columns={"reporter_country": "partner_country", "partner_country": "reporter_country", "value": "imp_qty"}
    )

    how = "outer" if include_unmatched else "inner"
    merged = exp.merge(imp, on=["reporter_country", "partner_country", "item_code"], how=how)
    merged["exp_qty"] = merged["exp_qty"].fillna(0)
    merged["imp_qty"] = merged["imp_qty"].fillna(0)
    if include_unmatched:
        merged = merged[(merged["exp_qty"] > 0) | (merged["imp_qty"] > 0)].copy()
    else:
        merged = merged[(merged["exp_qty"] > 0) & (merged["imp_qty"] > 0)].copy()

    exp_arr = merged["exp_qty"].to_numpy()
    imp_arr = merged["imp_qty"].to_numpy()
    max_arr = np.maximum(exp_arr, imp_arr)
    merged["agreement"] = np.minimum(exp_arr, imp_arr) / max_arr

    matched_band_labels = ["<25%", "25–50%", "50–75%", "75–90%", "≥90%"]
    matched_band_colors = ["#d73027", "#fc8d59", "#fee08b", "#91cf60", "#1a9850"]
    band = pd.cut(merged["agreement"], bins=[-0.001, 0.25, 0.50, 0.75, 0.90, 1.001], labels=matched_band_labels)
    if include_unmatched:
        unmatched_mask = (merged["exp_qty"] == 0) | (merged["imp_qty"] == 0)
        band = band.cat.add_categories("Unmatched")
        band[unmatched_mask] = "Unmatched"
        band_labels = ["Unmatched"] + matched_band_labels
        band_colors = ["#67000d"] + matched_band_colors
    else:
        band_labels = matched_band_labels
        band_colors = matched_band_colors
    merged["band"] = band

    top = merged["reporter_country"].value_counts().head(top_n).index.tolist()
    sub = merged[merged["reporter_country"].isin(top)]
    shares = sub.groupby(["reporter_country", "band"], observed=True).size().unstack(fill_value=0)
    shares = shares.div(shares.sum(axis=1), axis=0)[band_labels]
    shares = shares.sort_values("≥90%", ascending=True)

    long = shares.reset_index().melt(id_vars="reporter_country", var_name="band", value_name="share")
    title = (
        "Quantity agreement (matched flows + unmatched as 0%)"
        if include_unmatched
        else "Quantity agreement among matched flows"
    )
    x_label = "Share of all flows" if include_unmatched else "Share of matched flows"
    fig = px.bar(
        long,
        x="share",
        y="reporter_country",
        color="band",
        orientation="h",
        title=title,
        labels={"share": x_label, "reporter_country": "Reporter"},
        category_orders={"band": band_labels, "reporter_country": shares.index.tolist()},
        color_discrete_sequence=band_colors,
    )
    fig.update_layout(barmode="stack", xaxis_tickformat=".0%")
    fig.show()


plot_quantity_mismatch_by_reporter(tb, year=2023, top_n=20, include_unmatched=False)

### 4.1 Including unmatched flows

In [11]:
plot_quantity_mismatch_by_reporter(tb, year=2023, top_n=20, include_unmatched=True)